# Model Training for Charpy Temperature Prediction without using PCA Data

Train and evaluate multiple regression models to predict Charpy Temperature (°C) from original (non-PCA) features.
This analysis helps understand how material properties influence the testing temperature, a critical parameter for assessing material toughness in various service conditions.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import joblib
import os 
from sklearn.metrics import mean_squared_error, r2_score , mean_absolute_error
from sklearn.impute import KNNImputer

In [2]:
os.makedirs('trained_models', exist_ok=True)

In [3]:
df = pd.read_csv('../../welddatabase/welddb_new.csv')
print(f"Dataset shape: {df.shape}")

Dataset shape: (1652, 52)


In [4]:
df.columns

Index(['Carbon_%', 'Silicon_%', 'Manganese_%', 'Sulphur_%', 'Phosphorus_%',
       'Nickel_%', 'Chromium_%', 'Molybdenum_%', 'Vanadium_%', 'Copper_%',
       'Cobalt_%', 'Tungsten_%', 'Oxygen_weight%', 'Titanium_weight%',
       'Nitrogen_weight%', 'Aluminium_weight%', 'Boron_weight%',
       'Niobium_weight%', 'Tin_weight%', 'Arsenic_weight%', 'Antimony_weight%',
       'Interpass_Temp_C', 'PWHT_Temp_C', 'PWHT_Time_hours',
       'Yield_Strength_MPa', 'UTS_MPa', 'Elongation_%', 'Reduction_Area_%',
       'Charpy_Temp_C', 'Charpy_Energy_J', 'Hardness_kg_mm2', 'FATT_50%',
       'Primary_Ferrite_%', 'Ferrite_2nd_Phase_%', 'Acicular_Ferrite_%',
       'Martensite_%', 'Ferrite_Carbide_%', 'Electrode_Polarity_+',
       'Electrode_Polarity_-', 'Electrode_Polarity_0', 'Power_W',
       'Weld_Type_FCA', 'Weld_Type_GMAA', 'Weld_Type_GTAA', 'Weld_Type_MMA',
       'Weld_Type_NGGMA', 'Weld_Type_NGSAW', 'Weld_Type_SA', 'Weld_Type_SAA',
       'Weld_Type_ShMA', 'Weld_Type_TSA', 'Heat_Input_J_mm']

In [5]:
X = df.drop('Charpy_Temp_C', axis=1)
y = df['Charpy_Temp_C']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 1321
Testing samples: 331


In [6]:
# --- 1️⃣ Supprimer les lignes dont la cible est manquante ---
df_clean = df.dropna(subset=['Charpy_Temp_C'])
print(f"✅ Lignes conservées après suppression des NaN dans la cible : {len(df_clean)} / {len(df)}")

# --- 2️⃣ Supprimer les lignes trop incomplètes ---
df_filtered = df_clean[df_clean.isnull().mean(axis=1) < 0.5]
print(f"✅ Lignes restantes après filtrage : {len(df_filtered)}")

# --- 3️⃣ Séparer X et y ---
X = df_filtered.drop(columns=['Charpy_Temp_C'])
y = df_filtered['Charpy_Temp_C']

# --- 4️⃣ Séparer les colonnes numériques ---
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
print(f"📊 Colonnes numériques : {len(numeric_cols)}")

# --- 5️⃣ Appliquer KNNImputer uniquement sur ces colonnes ---
imputer = KNNImputer(n_neighbors=5, weights='distance')
X_num_imputed = imputer.fit_transform(X[numeric_cols])

# --- 6️⃣ Corriger automatiquement la cohérence de taille ---
valid_col_count = X_num_imputed.shape[1]
numeric_cols = numeric_cols[:valid_col_count]  # Ajuste si une colonne a été ignorée

# --- 7️⃣ Reconstruction DataFrame cohérent ---
X_imputed = pd.DataFrame(X_num_imputed, columns=numeric_cols, index=X.index)

print(f"✅ Imputation réussie. Dimensions finales : {X_imputed.shape}")
print(f"Nombre total de valeurs manquantes restantes : {X_imputed.isna().sum().sum()}")
print(f"Nombre de NaN dans y : {y.isna().sum()}")


✅ Lignes conservées après suppression des NaN dans la cible : 879 / 1652
✅ Lignes restantes après filtrage : 879
📊 Colonnes numériques : 51
✅ Imputation réussie. Dimensions finales : (879, 50)
Nombre total de valeurs manquantes restantes : 0
Nombre de NaN dans y : 0


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42
)

print(f"📊 Train set: {X_train.shape[0]} samples, Test set: {X_test.shape[0]} samples")


📊 Train set: 703 samples, Test set: 176 samples


In [8]:
# --- Define all regression models ---
models = {
    'LinearRegression': LinearRegression(),
    'RidgeRegression': Ridge(),
    'LassoRegression': Lasso(),
    'ElasticNetRegression': ElasticNet(),
    'DecisionTreeRegressor': DecisionTreeRegressor(random_state=42),
    'RandomForestRegressor': RandomForestRegressor(random_state=42, n_jobs=-1)
  
}

# --- Define the hyperparameter grids for each model ---
param_grids = {
    'LinearRegression': {},  # No hyperparameters to tune
    'RidgeRegression': {'alpha': [0.1, 1.0, 10.0, 100.0]},
    'LassoRegression': {'alpha': [0.1, 1.0, 10.0, 100.0]},
    'ElasticNetRegression': {'alpha': [0.1, 1.0, 10.0], 'l1_ratio': [0.2, 0.5, 0.8]},
    'DecisionTreeRegressor': {'max_depth': [5, 10, 20, None], 'min_samples_split': [2, 10, 20]},
    'RandomForestRegressor': {'n_estimators': [100, 200], 'max_depth': [10, 20, None]},
    'GradientBoostingRegressor': {'n_estimators': [100, 200], 'learning_rate': [0.01, 0.1, 0.2], 'max_depth': [3, 5, 10]}
    
}


In [10]:
# --- 2️⃣ Model training and evaluation ---
results = []

for name, model in models.items():
    print(f"\n🚀 Training {name}...")

    grid_search = GridSearchCV(
        model,
        param_grids[name],
        cv=5,
        scoring='r2',
        n_jobs=-1,
        verbose=0
    )

    # --- Train on the imputed dataset ---
    grid_search.fit(X_train, y_train)

    # --- Predictions on training and testing sets ---
    y_train_pred = grid_search.predict(X_train)
    y_test_pred = grid_search.predict(X_test)

    # --- Compute metrics ---
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    # --- Store model results ---
    results.append({
        'Model': name,
        'Best_Params': str(grid_search.best_params_),
        'CV_R2': grid_search.best_score_,
        'Train_R2': train_r2,
        'Test_R2': test_r2,
        'MAE_Test': test_mae,
        'RMSE_Test': test_rmse
    })

    # --- Save the best model with clear naming ---
    model_filename = f"trained_models/{name}_without_PCA_model.pkl"
    joblib.dump(grid_search.best_estimator_, model_filename)

    # --- Display quick summary ---
    print(f"✅ Best Parameters: {grid_search.best_params_}")
    print(f"CV R²: {grid_search.best_score_:.4f} | Train R²: {train_r2:.4f} | Test R²: {test_r2:.4f}")
    print(f"MAE: {test_mae:.4f} | RMSE: {test_rmse:.4f}")
    print(f"📁 Model saved as: {model_filename}")

print("\n🏁 Training completed for all models (WITHOUT PCA)!")

# --- 3️⃣ Create and display organized results table ---
results_df = pd.DataFrame(results).sort_values(by='Test_R2', ascending=False).reset_index(drop=True)

print("\n📊 MODEL PERFORMANCE SUMMARY (WITHOUT PCA)")
print("=" * 90)
display(results_df.style.set_properties(**{
    'background-color': '#1e1e1e',
    'color': 'white',
    'border-color': 'gray',
    'text-align': 'center'
}).format({
    'CV_R2': "{:.4f}",
    'Train_R2': "{:.4f}",
    'Test_R2': "{:.4f}",
    'MAE_Test': "{:.4f}",
    'RMSE_Test': "{:.4f}"
}))



🚀 Training LinearRegression...
✅ Best Parameters: {}
CV R²: 0.5917 | Train R²: 0.6812 | Test R²: 0.6492
MAE: 14.8751 | RMSE: 18.7422
📁 Model saved as: trained_models/LinearRegression_without_PCA_model.pkl

🚀 Training RidgeRegression...
✅ Best Parameters: {'alpha': 0.1}
CV R²: 0.5738 | Train R²: 0.6625 | Test R²: 0.6370
MAE: 15.4761 | RMSE: 19.0648
📁 Model saved as: trained_models/RidgeRegression_without_PCA_model.pkl

🚀 Training LassoRegression...


c:\Users\user\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.620e+04, tolerance: 8.821e+01
  model = cd_fast.enet_coordinate_descent(


✅ Best Parameters: {'alpha': 0.1}
CV R²: 0.5529 | Train R²: 0.6261 | Test R²: 0.6171
MAE: 15.9443 | RMSE: 19.5801
📁 Model saved as: trained_models/LassoRegression_without_PCA_model.pkl

🚀 Training ElasticNetRegression...


c:\Users\user\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.752e+04, tolerance: 8.821e+01
  model = cd_fast.enet_coordinate_descent(


✅ Best Parameters: {'alpha': 0.1, 'l1_ratio': 0.8}
CV R²: 0.5521 | Train R²: 0.6171 | Test R²: 0.6081
MAE: 16.0300 | RMSE: 19.8098
📁 Model saved as: trained_models/ElasticNetRegression_without_PCA_model.pkl

🚀 Training DecisionTreeRegressor...
✅ Best Parameters: {'max_depth': 20, 'min_samples_split': 20}
CV R²: 0.5596 | Train R²: 0.8486 | Test R²: 0.5822
MAE: 14.7831 | RMSE: 20.4519
📁 Model saved as: trained_models/DecisionTreeRegressor_without_PCA_model.pkl

🚀 Training RandomForestRegressor...
✅ Best Parameters: {'max_depth': None, 'n_estimators': 200}
CV R²: 0.7270 | Train R²: 0.9463 | Test R²: 0.7818
MAE: 11.3282 | RMSE: 14.7798
📁 Model saved as: trained_models/RandomForestRegressor_without_PCA_model.pkl

🏁 Training completed for all models (WITHOUT PCA)!

📊 MODEL PERFORMANCE SUMMARY (WITHOUT PCA)


,Model,Best_Params,CV_R2,Train_R2,Test_R2,MAE_Test,RMSE_Test
0,RandomForestRegressor,"{'max_depth': None, 'n_estimators': 200}",0.7270,0.9463,0.7818,11.3282,14.7798
1,LinearRegression,{},0.5917,0.6812,0.6492,14.8751,18.7422
2,RidgeRegression,{'alpha': 0.1},0.5738,0.6625,0.6370,15.4761,19.0648
3,LassoRegression,{'alpha': 0.1},0.5529,0.6261,0.6171,15.9443,19.5801
4,ElasticNetRegression,"{'alpha': 0.1, 'l1_ratio': 0.8}",0.5521,0.6171,0.6081,16.0300,19.8098
5,DecisionTreeRegressor,"{'max_depth': 20, 'min_samples_split': 20}",0.5596,0.8486,0.5822,14.7831,20.4519
